In [ ]:
import logging
from datetime import datetime
import torch
from engine import evaluate, evaluate_with_threshold
import utils
from instance_dataset import BoundingBoxDataset
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import csv

# Setup logging configuration
start_time_str = datetime.now().strftime("%Y%m%d_%H%M")
logging.basicConfig(
    filename=f"log_{start_time_str}.txt",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

# Log the beginning of the process
logging.info("Starting evaluation process.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using {device} for evaluating.")

# Data loading
test_dataset = BoundingBoxDataset("../../test_data", get_transforms(is_train=False))
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=4, shuffle=False, collate_fn=utils.collate_fn
)
logging.info("Test dataset loaded and DataLoader set up.")

# Model files to evaluate
model_files = [
    "faster-rcnn-20240427_2343.pth",
    "no-agar-faster-rcnn-20240429_0352.pth",
    "random-faster-rcnn-20240430_1531.pth",
]
logging.info("Model files to be evaluated: " + ", ".join(model_files))

# Initiate CSV writer
results_file = f"test_results_{start_time_str}.csv"
with open(results_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        [
            "model",
            "threshold",
            "avg_abs_error",
            "avg_rel_error",
            "AP_IoU_0.50_0.95_all_100",
            "AP_IoU_0.50_all_100",
            "AP_IoU_0.75_all_100",
            "AP_IoU_0.50_0.95_small_100",
            "AP_IoU_0.50_0.95_medium_100",
            "AP_IoU_0.50_0.95_large_100",
            "AR_IoU_0.50_0.95_all_1",
            "AR_IoU_0.50_0.95_all_10",
            "AR_IoU_0.50_0.95_all_100",
            "AR_IoU_0.50_0.95_small_100",
            "AR_IoU_0.50_0.95_medium_100",
            "AR_IoU_0.50_0.95_large_100",
        ]
    )
    logging.info("CSV file setup complete.")

    for model_file in model_files:
        model = get_model_instance_segmentation(2)
        model.load_state_dict(torch.load(model_file, map_location=device))
        logging.info(f"Loaded model from {model_file}")

        coco_evaluator, avg_abs_error, avg_rel_error = evaluate(
            model, test_loader, device=device
        )
        stats = coco_evaluator.coco_eval["bbox"].stats.tolist()
        writer.writerow([model_file, 0, avg_abs_error, avg_rel_error, *stats])
        logging.info(f"Evaluated model {model_file} with default threshold.")

        for threshold in [0.5, 0.8, 0.9]:
            coco_evaluator, avg_abs_error, avg_rel_error = evaluate_with_threshold(
                model, test_loader, device=device, threshold=threshold
            )
            stats = coco_evaluator.coco_eval["bbox"].stats.tolist()
            writer.writerow(
                [model_file, threshold, avg_abs_error, avg_rel_error, *stats]
            )
            logging.info(f"Evaluated model {model_file} with threshold {threshold}.")

logging.info("Evaluation process completed.")